In [2]:
import csv
from datetime import datetime
from pathlib import Path

CWD = Path.cwd()
if (CWD / "data:raw:").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "data:raw:").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError("Could not locate data:raw: folder from current working directory")

RAW_DIR = PROJECT_ROOT / "data:raw:"
CLEAN_DIR = PROJECT_ROOT / "data:clean:"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Range endpoints are inclusive.
DATASETS = [
    {
        "label": "Palworld",
        "filename": "Palworld.csv",
        "start": "2024-12-09",
        "end": "2025-01-06",
    },
    {
        "label": "HELLDIVERS 2",
        "filename": "Helldivers.csv",
        "start": "2025-08-19",
        "end": "2025-09-16",
        "update_date": "2025-09-02",
    },
    {
        "label": "No Man's Sky",
        "filename": "No mans sky.csv",
        "start": "2025-01-15",
        "end": "2025-02-12",
    },
    {
        "label": "Destiny 2",
        "filename": "Destiny 2.csv",
        "start": "2024-09-24",
        "end": "2024-10-22",
    },
    {
        "label": "Counter-Strike 2",
        "filename": "Counter strike 2.csv",
        "start": "2024-04-23",
        "end": "2024-05-21",
    },
    {
        "label": "Apex Legends",
        "filename": "Apex Legends.csv",
        "start": "2025-01-28",
        "end": "2025-02-25",
    },
    {
        "label": "Deep Rock Galactic",
        "filename": "Dead rock galactic.csv",
        "start": "2024-05-30",
        "end": "2024-06-27",
    },
    {
        "label": "Don't Starve Together",
        "filename": "Dont Starve.csv",
        "start": "2023-04-13",
        "end": "2023-05-11",
    },
    {
        "label": "Dead by Daylight",
        "filename": "Dead by Daylight.csv",
        "start": "2023-11-14",
        "end": "2023-12-12",
    },
    {
        "label": "PUBG: BATTLEGROUNDS",
        "filename": "PUBG.csv",
        "start": "2025-10-22",
        "end": "2025-11-19",
    },
    {
        "label": "Sea of Thieves",
        "filename": "Sea of Thieves.csv",
        "start": "2024-10-03",
        "end": "2024-10-31",
    },
    {
        "label": "Warframe",
        "filename": "Warframe.csv",
        "start": "2025-11-26",
        "end": "2025-12-24",
    },
]


def sanitize_name(name: str) -> str:
    allowed = []
    for ch in name.lower():
        if ch.isalnum():
            allowed.append(ch)
        else:
            allowed.append("_")
    return "".join(allowed).strip("_")


def filter_csv_by_date(source_file: Path, output_file: Path, start_iso: str, end_iso: str) -> int:
    start_date = datetime.strptime(start_iso, "%Y-%m-%d").date()
    end_date = datetime.strptime(end_iso, "%Y-%m-%d").date()

    with source_file.open("r", newline="", encoding="utf-8-sig") as src, output_file.open(
        "w", newline="", encoding="utf-8"
    ) as dst:
        reader = csv.DictReader(src)
        writer = csv.DictWriter(dst, fieldnames=reader.fieldnames)
        writer.writeheader()

        rows_written = 0
        for row in reader:
            row_date = datetime.strptime(row["DateTime"], "%Y-%m-%d %H:%M:%S").date()
            if start_date <= row_date <= end_date:
                writer.writerow(row)
                rows_written += 1

    return rows_written


results = []
for dataset in DATASETS:
    source_path = RAW_DIR / dataset["filename"]
    output_name = (
        f"{sanitize_name(dataset['label'])}_{dataset['start']}_to_{dataset['end']}.csv"
    )
    output_path = CLEAN_DIR / output_name

    count = filter_csv_by_date(
        source_file=source_path,
        output_file=output_path,
        start_iso=dataset["start"],
        end_iso=dataset["end"],
    )

    result_line = f"{dataset['label']}: wrote {count} rows -> {output_path}"
    if "update_date" in dataset:
        result_line += f" (update date: {dataset['update_date']})"
    results.append(result_line)

for line in results:
    print(line)

Palworld: wrote 29 rows -> /Users/timothytran/Documents/Python/clone/ecc3479-project/data:clean:/palworld_2024-12-09_to_2025-01-06.csv
HELLDIVERS 2: wrote 29 rows -> /Users/timothytran/Documents/Python/clone/ecc3479-project/data:clean:/helldivers_2_2025-08-19_to_2025-09-16.csv (update date: 2025-09-02)
No Man's Sky: wrote 29 rows -> /Users/timothytran/Documents/Python/clone/ecc3479-project/data:clean:/no_man_s_sky_2025-01-15_to_2025-02-12.csv
Destiny 2: wrote 29 rows -> /Users/timothytran/Documents/Python/clone/ecc3479-project/data:clean:/destiny_2_2024-09-24_to_2024-10-22.csv
Counter-Strike 2: wrote 29 rows -> /Users/timothytran/Documents/Python/clone/ecc3479-project/data:clean:/counter_strike_2_2024-04-23_to_2024-05-21.csv
Apex Legends: wrote 29 rows -> /Users/timothytran/Documents/Python/clone/ecc3479-project/data:clean:/apex_legends_2025-01-28_to_2025-02-25.csv
Deep Rock Galactic: wrote 29 rows -> /Users/timothytran/Documents/Python/clone/ecc3479-project/data:clean:/deep_rock_galac